In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra

import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ieee-fraud-detection/sample_submission.csv
/kaggle/input/ieee-fraud-detection/test_identity.csv
/kaggle/input/ieee-fraud-detection/train_identity.csv
/kaggle/input/ieee-fraud-detection/test_transaction.csv
/kaggle/input/ieee-fraud-detection/train_transaction.csv


In [5]:
!pip install mlflow dagshub --quiet

In [12]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import RFE
from scipy import stats
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import datetime
from datetime import timedelta

# ML models
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier

# For handling imbalanced data
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

In [6]:
!pip install mlflow dagshub --quiet
import mlflow
from dagshub import dagshub_logger
import os

# Set tracking URI manually
mlflow.set_tracking_uri("https://dagshub.com/ekvirika/FraudDerection.mlflow")

# Use your DagsHub credentials
os.environ["MLFLOW_TRACKING_USERNAME"] = "ekvirika"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "3f601f2c2c7a6bca448ebc69f4f5d4b49daffd8f"

# Optional: set registry if you're using model registry
mlflow.set_registry_uri("https://dagshub.com/ekvirika/FraudDerection.mlflow")

In [8]:
import mlflow
mlflow.set_experiment("RandomForest_Training")

<Experiment: artifact_location='mlflow-artifacts:/32ab6f483e4841628736b89cb8eb10c0', creation_time=1744702177290, experiment_id='3', last_update_time=1744702177290, lifecycle_stage='active', name='RandomForest_Training', tags={}>

Load file from mlflow experiments with user_id

In [ ]:
df = pd.read_csv('/kaggle/input/ieee-fraud-detection/train_transaction.csv')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.sklearn
import xgboost as xgb
from category_encoders import WOEEncoder
import warnings
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from imblearn.pipeline import Pipeline as ImbPipeline  # ✅ Corrected import
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
import shap
from sklearn.ensemble import RandomForestClassifier
import mlflow.data
from mlflow.data.pandas_dataset import PandasDataset
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns


warnings.filterwarnings('ignore')

# Assume df is already loaded and merged
mlflow.set_experiment('RandomForest_Training')

with mlflow.start_run(run_name="RandomForest_GridSearch"):

    y = df['isFraud']
    X = df.drop(columns=['isFraud', 'TransactionID'])

    # Drop high-missing columns
    missing_ratio = X.isnull().mean()
    cols_to_drop = missing_ratio[missing_ratio > 0.9].index.tolist()
    X.drop(columns=cols_to_drop, inplace=True)
    mlflow.log_param("dropped_cols_90pct_na", len(cols_to_drop))

    # Identify types
    cat_cols = X.select_dtypes(include='object').columns.tolist()
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

    mlflow.log_param("categorical_features", len(cat_cols))
    mlflow.log_param("numerical_features", len(num_cols))

    # Train-test split by user
    user_list = df["user_id"].unique()
    train_users, valid_users = train_test_split(user_list, test_size=0.2, random_state=42)

    train_mask = df["user_id"].isin(train_users)
    valid_mask = df["user_id"].isin(valid_users)

    X_train = df[train_mask].drop(columns=["isFraud"])
    y_train = df[train_mask]["isFraud"]
    X_valid = df[valid_mask].drop(columns=["isFraud"])
    y_valid = df[valid_mask]["isFraud"]

    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", WOEEncoder())
    ])
    preprocessor = ColumnTransformer([
        ("num", num_pipeline, num_cols),
        ("cat", cat_pipeline, cat_cols)
    ])

    full_pipeline = ImbPipeline([
        ("preprocessor", preprocessor),
        ("sampler", RandomUnderSampler(random_state=42)),  # 👈 Replaced SMOTE
        ("clf", RandomForestClassifier(random_state=42))
    ])

    
    param_grid = {
        "clf__n_estimators": [120, 150, 200],
        "clf__max_depth": [11, 12, 15],
        "clf__class_weight": ["balanced"]
    }


    search = GridSearchCV(
        full_pipeline,
        param_grid,
        scoring='roc_auc',
        cv=2,  # ⏱️ faster
        verbose=1,
        n_jobs=-1  # ⏱️ parallel processing
    )

    search.fit(X_train, y_train)
    best_model = search.best_estimator_
    best_params = search.best_params_
    
    # Log best params
    for param, val in best_params.items():
        mlflow.log_param(param, val)
    
    # Predict and evaluate
    y_pred = best_model.predict(X_valid)
    y_pred_proba = best_model.predict_proba(X_valid)[:, 1]
    
    # Compute metrics
    auc = roc_auc_score(y_valid, y_pred_proba)
    accuracy = accuracy_score(y_valid, y_pred)
    precision = precision_score(y_valid, y_pred, zero_division=0)
    recall = recall_score(y_valid, y_pred, zero_division=0)
    f1 = f1_score(y_valid, y_pred, zero_division=0)
    cm = confusion_matrix(y_valid, y_pred)
    
    # Log metrics
    mlflow.log_metric("val_auc", auc)
    mlflow.log_metric("val_accuracy", accuracy)
    mlflow.log_metric("val_precision", precision)
    mlflow.log_metric("val_recall", recall)
    mlflow.log_metric("val_f1", f1)
    
    # Log confusion matrix as image
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig("confusion_matrix.png")
    mlflow.log_artifact("confusion_matrix.png")
    
    # Save the model
    mlflow.sklearn.log_model(best_model, "RandomForest_pipeline")
    
    print(f"Best AUC: {auc:.4f}")
    print("Best Parameters:")
    print(best_params)


In [10]:
!pip install imbalanced-learn==0.11.0 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.6/235.6 kB 9.2 MB/s eta 0:00:00


In [31]:
class ColumnDropper(BaseEstimator, TransformerMixin):
    """Drop columns with too many missing values"""
    def __init__(self, null_threshold=0.8):
        self.null_threshold = null_threshold
        self.cols_to_drop = None
        
    def fit(self, X, y=None):
        # Calculate percentage of missing values per column
        null_percentages = X.isnull().mean()
        # Identify columns to drop
        self.cols_to_drop = null_percentages[null_percentages > self.null_threshold].index.tolist()
        return self
        
    def transform(self, X):
        # Drop columns with too many missing values
        return X.drop(columns=self.cols_to_drop, errors='ignore')


class MissingValueImputer(BaseEstimator, TransformerMixin):
    """Impute missing values"""
    def __init__(self):
        self.num_impute_values = {}
        self.cat_impute_values = {}
        
    def fit(self, X, y=None):
        # For numeric columns: median
        numeric_cols = X.select_dtypes(include=['float64', 'int64']).columns
        for col in numeric_cols:
            if X[col].isna().any():
                self.num_impute_values[col] = X[col].median()
        
        # For categorical columns: most frequent value
        cat_cols = X.select_dtypes(include=['object', 'category']).columns
        for col in cat_cols:
            if X[col].isna().any():
                self.cat_impute_values[col] = X[col].value_counts().index[0]
                
        return self
        
    def transform(self, X):
        X_transformed = X.copy()
        
        # Impute numeric values
        for col, value in self.num_impute_values.items():
            if col in X_transformed.columns:
                X_transformed[col] = X_transformed[col].fillna(value)
        
        # Impute categorical values
        for col, value in self.cat_impute_values.items():
            if col in X_transformed.columns:
                X_transformed[col] = X_transformed[col].fillna(value)
                
        return X_transformed


class TimeFeatureExtractor(BaseEstimator, TransformerMixin):
    """Extract time-based features"""
    def __init__(self, time_col='TransactionDT'):
        self.time_col = time_col
        
    def fit(self, X, y=None):
        return self
        
    def transform(self, X):
        X_transformed = X.copy()
        
        if self.time_col in X_transformed.columns:
            # Convert to seconds (assuming TransactionDT is in seconds)
            time_secs = X_transformed[self.time_col]
            
            # Create time features (assuming TransactionDT is seconds from reference)
            X_transformed['hour_of_day'] = ((time_secs / 3600) % 24).astype(int)
            X_transformed['day_of_week'] = ((time_secs / (3600 * 24)) % 7).astype(int)
            X_transformed['day_part'] = X_transformed['hour_of_day'].apply(
                lambda x: 'morning' if 5 <= x < 12 else 
                           'afternoon' if 12 <= x < 17 else
                           'evening' if 17 <= x < 21 else 'night'
            )
        
        return X_transformed


class UserBehaviorFeatureExtractor(BaseEstimator, TransformerMixin):
    """Extract user behavior features based on historical patterns"""
    
    def __init__(self, use_multiple_ids=False, user_col='card1'):
        self.use_multiple_ids = use_multiple_ids
        self.user_col = user_col
        # Initialize dictionaries for all required statistics
        self.user_stats = {
            'tx_count': {},
            'fraud_rate': {},
            'fraud_count': {}
        }
    
    def fit(self, X, y=None):
        # Check if user column exists
        if self.user_col not in X.columns:
            # If the column doesn't exist, we'll just return without setting stats
            return self
            
        # Get user transaction counts
        user_counts = X[self.user_col].value_counts().to_dict()
        self.user_stats['tx_count'] = user_counts
        
        if y is not None:
            # Create a temporary DataFrame with user_id and fraud label
            temp_df = pd.DataFrame({self.user_col: X[self.user_col], 'isFraud': y})
            
            # Calculate fraud counts per user
            fraud_df = temp_df[temp_df['isFraud'] == 1]
            if not fraud_df.empty:
                fraud_counts = fraud_df[self.user_col].value_counts().to_dict()
            else:
                fraud_counts = {}
                
            self.user_stats['fraud_count'] = fraud_counts
            
            # Calculate fraud rates
            for user_id in user_counts:
                fraud_count = fraud_counts.get(user_id, 0)
                tx_count = user_counts[user_id]
                self.user_stats['fraud_rate'][user_id] = fraud_count / tx_count if tx_count > 0 else 0
            
        return self
    
    def transform(self, X):
        X_transformed = X.copy()
        
        # Make sure the user column exists
        if self.user_col not in X_transformed.columns:
            # Add placeholder features if user column doesn't exist
            X_transformed['user_tx_count'] = 1  # Default
            X_transformed['user_fraud_rate'] = 0  # Default
            X_transformed['user_fraud_count'] = 0  # Default
            return X_transformed
            
        # Add user behavior features with safe mapping using .get() method
        X_transformed['user_tx_count'] = X_transformed[self.user_col].map(
            lambda x: self.user_stats['tx_count'].get(x, 1)
        ).fillna(1)
        
        X_transformed['user_fraud_rate'] = X_transformed[self.user_col].map(
            lambda x: self.user_stats['fraud_rate'].get(x, 0)
        ).fillna(0)
        
        X_transformed['user_fraud_count'] = X_transformed[self.user_col].map(
            lambda x: self.user_stats['fraud_count'].get(x, 0)
        ).fillna(0)
            
        return X_transformed


class CategoricalEncoder(BaseEstimator, TransformerMixin):
    """Encode categorical variables"""
    def __init__(self, threshold=10):
        self.threshold = threshold
        self.encoding_maps = {}
        self.rare_values = {}
        
    def fit(self, X, y=None):
        # Find categorical columns
        cat_cols = X.select_dtypes(include=['object', 'category']).columns
        
        for col in cat_cols:
            # Count frequencies
            value_counts = X[col].value_counts()
            
            # Identify rare values (less than threshold occurrences)
            self.rare_values[col] = value_counts[value_counts < self.threshold].index.tolist()
            
            # Create encoding map for each value (exclude rare values)
            unique_values = [val for val in X[col].unique() if val not in self.rare_values[col] and pd.notna(val)]
            self.encoding_maps[col] = {val: idx for idx, val in enumerate(unique_values, 1)}
            
        return self
    
    def transform(self, X):
        X_transformed = X.copy()
        
        # Apply encoding to categorical columns
        for col, encoding_map in self.encoding_maps.items():
            if col in X_transformed.columns:
                # Replace rare values with 'rare'
                for rare_val in self.rare_values[col]:
                    X_transformed.loc[X_transformed[col] == rare_val, col] = 'rare'
                
                # Map values using encoding
                X_transformed[col] = X_transformed[col].map(
                    lambda x: encoding_map.get(x, 0) if pd.notna(x) else 0
                )
            
        return X_transformed


class NumericalProcessor(BaseEstimator, TransformerMixin):
    """Process numerical variables: scaling and outlier handling"""
    def __init__(self, z_threshold=3.0):
        self.z_threshold = z_threshold
        self.scaler = StandardScaler()
        self.numeric_cols = None
        
    def fit(self, X, y=None):
        # Find numeric columns
        self.numeric_cols = X.select_dtypes(include=['float64', 'int64']).columns.tolist()
        
        # Fit scaler on numeric columns
        if self.numeric_cols:
            self.scaler.fit(X[self.numeric_cols])
            
        return self
    
    def transform(self, X):
        X_transformed = X.copy()
        
        # Scale numeric columns
        if self.numeric_cols:
            # Get columns that exist in the input data
            existing_cols = [col for col in self.numeric_cols if col in X_transformed.columns]
            
            if existing_cols:
                # Scale existing numeric columns
                X_scaled = self.scaler.transform(X_transformed[existing_cols])
                
                # Handle outliers
                X_scaled = np.clip(X_scaled, -self.z_threshold, self.z_threshold)
                
                # Replace original columns with scaled values
                X_transformed[existing_cols] = X_scaled
            
        return X_transformed

# Pipeline

In [32]:
def create_preprocessing_pipeline():
    """Create the preprocessing pipeline with fixed components"""
    preprocessing = Pipeline([
        ('drop_na', ColumnDropper(null_threshold=0.8)),
        ('imputer', MissingValueImputer()),
        ('time_extractor', TimeFeatureExtractor()),
        ('user_behavior_extractor', UserBehaviorFeatureExtractor(use_multiple_ids=False, user_col='card1')),
        ('cat_encoder', CategoricalEncoder(threshold=3)),
        ('num_processor', NumericalProcessor(z_threshold=3.5))
    ])
    
    return preprocessing


def create_training_pipeline():
    """Create pipeline for target-dependent encoding and additional features"""
    training_pipe = Pipeline([
        # Add any target-dependent transformations here
    ])
    
    return training_pipe

In [33]:
def timestamp_train_test_split(X, y, test_size=0.2, time_col='TransactionDT'):
    """Split data by timestamp to prevent data leakage"""
    if time_col in X.columns:
        # Sort by timestamp
        sorted_indices = X[time_col].argsort()
        X_sorted = X.iloc[sorted_indices]
        y_sorted = y.iloc[sorted_indices]
        
        # Calculate split point
        split_idx = int(len(X) * (1 - test_size))
        
        # Split data
        X_train = X_sorted.iloc[:split_idx]
        X_test = X_sorted.iloc[split_idx:]
        y_train = y_sorted.iloc[:split_idx]
        y_test = y_sorted.iloc[split_idx:]
    else:
        # Fallback to random split if time column not available
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42
        )
    
    return X_train, X_test, y_train, y_test

In [34]:
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression  # For RFE selector
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """Evaluate model and generate performance metrics"""
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score, f1_score,
        roc_auc_score, confusion_matrix, precision_recall_curve,
        roc_curve, auc
    )
    import matplotlib.pyplot as plt
    
    # Make predictions
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_proba)
    }
    
    # Generate confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title(f'Confusion Matrix - {model_name}')
    plt.colorbar()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.savefig(f"{model_name.lower().replace(' ', '_')}_confusion_matrix.png")
    
    # Generate ROC curve
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f'ROC curve (area = {metrics["roc_auc"]:.3f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve - {model_name}')
    plt.legend(loc='lower right')
    plt.savefig(f"{model_name.lower().replace(' ', '_')}_roc_curve.png")
    
    # Generate PR curve
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, label=f'PR curve')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'Precision-Recall Curve - {model_name}')
    plt.legend(loc='lower left')
    plt.savefig(f"{model_name.lower().replace(' ', '_')}_pr_curve.png")
    
    return metrics

In [36]:
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

def main():
    # ⚡ Load data
    print("Loading data...")
    df = pd.read_csv('/kaggle/input/ieee-fraud-detection/train_transaction.csv')
    y = df['isFraud']
    
    # 🏗️ Feature engineering
    print("Applying preprocessing pipeline...")
    preprocessor = create_preprocessing_pipeline()
    X_processed = preprocessor.fit_transform(df)
    
    # ⏳ Train-test split by timestamp
    print("Splitting data by timestamp...")
    X_train, X_test, y_train, y_test = timestamp_train_test_split(X_processed, y)
    
    # Log data split sizes
    print(f"Training set: {X_train.shape[0]} samples")
    print(f"Test set: {X_test.shape[0]} samples")
    
    # 🎯 Encode target-dependent features
    print("Applying training pipeline...")
    training_pipe = create_training_pipeline()
    X_train_encoded = training_pipe.fit_transform(X_train, y_train)
    X_test_encoded = training_pipe.transform(X_test)
    
    # 🧪 Feature selection via RFE
    print("Selecting features with RFE...")
    rfe_selector = RFE(
        estimator=LogisticRegression(max_iter=1000, C=0.1, solver='liblinear'), 
        n_features_to_select=50, 
        step=10,
        verbose=1
    )
    X_train_rfe = rfe_selector.fit_transform(X_train_encoded, y_train)
    X_test_rfe = rfe_selector.transform(X_test_encoded)
    
    # Log selected features
    selected_features = [
        col for col, selected in zip(X_train_encoded.columns, rfe_selector.support_) if selected
    ]
    print(f"Selected {len(selected_features)} features: {selected_features[:5]}...")
    
    # 🧬 Balance data with SMOTE
    print("Balancing data with SMOTE...")
    smote = SMOTE(random_state=42, n_jobs=-1)
    X_train_bal, y_train_bal = smote.fit_resample(X_train_rfe, y_train)
    
    # Log class distribution
    print(f"Class distribution after SMOTE: {pd.Series(y_train_bal).value_counts()}")
    
    # 🌲 Train Random Forest
    print("Training Random Forest model...")
    rf = RandomForestClassifier(
        n_estimators=100, 
        max_depth=10, 
        min_samples_split=10,
        min_samples_leaf=4,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train_bal, y_train_bal)
    
    # 🧪 Evaluate + Log via MLflow
    print("Evaluating model and logging to MLflow...")
    with mlflow.start_run(run_name="RandomForest_Training"):
        model_name = "Random Forest (RFE + SMOTE)"
        
        # 📈 Evaluate
        metrics = evaluate_model(rf, X_train_bal, X_test_rfe, y_train_bal, y_test, model_name)
        
        # Log metrics
        for key, value in metrics.items():
            mlflow.log_metric(key, value)
            print(f"{key}: {value:.4f}")
        
        # Log plots
        mlflow.log_artifact("random_forest_confusion_matrix.png")
        mlflow.log_artifact("random_forest_roc_curve.png")
        mlflow.log_artifact("random_forest_pr_curve.png")
        
        # Log model
        mlflow.sklearn.log_model(
            sk_model=rf,
            artifact_path="random_forest_model",
            registered_model_name="FraudDetection_RF_RFE_SMOTE"
        )
        
        print("\n✅ Model trained, evaluated, and logged successfully with RFE + SMOTE")

if __name__ == "__main__":
    main()

Loading data...
Applying preprocessing pipeline...
Splitting data by timestamp...
Training set: 472432 samples
Test set: 118108 samples
Applying training pipeline...


ValueError: not enough values to unpack (expected 2, got 0)